# 04. 실제 리뷰 속성·감성 테스트

이 노트북에서는 두 가지 방법으로 테스트합니다.

1. 속성을 직접 지정한 문장의 긍정·부정·중립 분석
2. 리뷰 문장만 여러 개 입력하여 속성과 감성을 연속으로 분석

위에서 아래로 한 번씩 실행하면 됩니다.

## 1. 모델 불러오기

- 속성 모델: 문장 → 26개 속성 중 하나
- 감성 모델: 속성 + 문장 → 긍정·부정·중립 중 하나

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

root_candidates = [Path.cwd(), *Path.cwd().parents, Path("/home/bteam/aspect_sentiment")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "aspect_labels.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("aspect_labels.json을 찾을 수 없습니다. 프로젝트 폴더 안에서 실행하세요.")

settings = json.loads((PROJECT_ROOT / "aspect_labels.json").read_text(encoding="utf-8"))
ASPECT_MODEL_DIR = PROJECT_ROOT / settings["model_output_dir"]
SENTIMENT_MODEL_DIR = PROJECT_ROOT / "models" / "sentiment"
MAX_LENGTH = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

aspect_tokenizer = AutoTokenizer.from_pretrained(ASPECT_MODEL_DIR)
aspect_model = AutoModelForSequenceClassification.from_pretrained(
    ASPECT_MODEL_DIR
).to(device)
aspect_model.eval()

sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_DIR)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    SENTIMENT_MODEL_DIR
).to(device)
sentiment_model.eval()

AVAILABLE_ASPECTS = set(aspect_model.config.id2label.values())

print("사용 장치:", device)
print("속성 모델:", ASPECT_MODEL_DIR)
print("감성 모델:", SENTIMENT_MODEL_DIR)
print("사용 가능한 속성:", len(AVAILABLE_ASPECTS), "개")

## 2. 예측 함수

`predict_aspect()`는 문장의 속성을 찾고, `predict_sentiment()`는 주어진 속성을 기준으로 문장의 감성을 찾습니다.

`analyze_review()`는 두 함수를 순서대로 실행합니다.

In [ ]:
def predict_aspect(text):
    text = str(text).strip()
    if not text:
        raise ValueError("빈 문장은 분석할 수 없습니다.")

    inputs = aspect_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    )
    inputs = {name: value.to(device) for name, value in inputs.items()}

    with torch.inference_mode():
        logits = aspect_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(probabilities.argmax())
    top_count = min(3, len(probabilities))
    top_probabilities, top_ids = torch.topk(probabilities, k=top_count)
    top3 = " / ".join(
        f"{aspect_model.config.id2label[int(label_id)]} {float(probability):.1%}"
        for probability, label_id in zip(top_probabilities, top_ids)
    )
    return {
        "aspect": aspect_model.config.id2label[predicted_id],
        "aspect_confidence": float(probabilities[predicted_id]),
        "aspect_top3": top3,
    }


def predict_sentiment(aspect, text):
    aspect = str(aspect).strip()
    text = str(text).strip()
    if aspect not in AVAILABLE_ASPECTS:
        raise ValueError(f"모델에 없는 속성입니다: {aspect}")
    if not text:
        raise ValueError("빈 문장은 분석할 수 없습니다.")

    model_input = f"[속성] {aspect} [문장] {text}"
    inputs = sentiment_tokenizer(
        model_input,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    )
    inputs = {name: value.to(device) for name, value in inputs.items()}

    with torch.inference_mode():
        logits = sentiment_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(probabilities.argmax())
    probability_text = " / ".join(
        f"{sentiment_model.config.id2label[label_id]} {float(probability):.1%}"
        for label_id, probability in enumerate(probabilities)
    )
    return {
        "sentiment": sentiment_model.config.id2label[predicted_id],
        "sentiment_confidence": float(probabilities[predicted_id]),
        "sentiment_probabilities": probability_text,
    }


def analyze_review(text):
    aspect_result = predict_aspect(text)
    sentiment_result = predict_sentiment(aspect_result["aspect"], text)
    return {
        "리뷰": text,
        "예측 속성": aspect_result["aspect"],
        "속성 신뢰도": f'{aspect_result["aspect_confidence"]:.1%}',
        "속성 Top 3": aspect_result["aspect_top3"],
        "예측 감성": sentiment_result["sentiment"],
        "감성 신뢰도": f'{sentiment_result["sentiment_confidence"]:.1%}',
        "감성별 확률": sentiment_result["sentiment_probabilities"],
    }


print("예측 함수 준비 완료")

## 3. 속성을 직접 지정하여 감성 테스트

이미 속성별로 문장이 분리되어 있다면 아래 목록의 `aspect`와 `sentence`만 바꾸면 됩니다.

이 테스트에서는 속성 모델을 거치지 않고 감성 모델만 실행합니다.

In [ ]:
aspect_sentences = [
    {"aspect": "발림성", "sentence": "로션처럼 부드럽고 고르게 잘 발려요."},
    {"aspect": "향", "sentence": "향이 너무 강해서 사용하기 부담스러워요."},
    {"aspect": "보습력/수분감", "sentence": "촉촉함은 보통이고 특별히 건조하지도 않아요."},
]

sentiment_results = []
for item in aspect_sentences:
    result = predict_sentiment(item["aspect"], item["sentence"])
    sentiment_results.append({
        "속성": item["aspect"],
        "문장": item["sentence"],
        "예측 감성": result["sentiment"],
        "감성 신뢰도": f'{result["sentiment_confidence"]:.1%}',
        "감성별 확률": result["sentiment_probabilities"],
    })

display(pd.DataFrame(sentiment_results))

## 4. 리뷰 여러 문장 자동 테스트

`reviews` 목록에 테스트할 리뷰 문장만 넣으면 됩니다.

각 문장에 대해 속성을 먼저 예측하고, 예측된 속성을 기준으로 감성까지 분석합니다. 한 문장에 여러 속성이 섞여 있으면 현재 속성 모델은 대표 속성 하나만 선택하므로 가능한 한 속성별로 나뉜 짧은 문장을 넣는 것이 좋습니다.

In [ ]:
reviews = [
    "촉촉하고 부드럽게 잘 발려요.",
    "향이 너무 강해서 사용하기 불편해요.",
    "가격은 조금 비싸지만 용량은 넉넉해요.",
    "오후가 되면 화장이 쉽게 무너져요.",
]

results = [analyze_review(review) for review in reviews]
display(pd.DataFrame(results))

## 결과 해석

- `속성 신뢰도`: 26개 속성 중 선택된 속성의 softmax 확률
- `속성 Top 3`: 가능성이 높았던 속성 세 개
- `감성 신뢰도`: 긍정·부정·중립 중 선택된 감성의 softmax 확률
- `감성별 확률`: 긍정·부정·중립 확률 전체

신뢰도가 높아도 반드시 정답이라는 뜻은 아닙니다. 특히 한 문장에 가격과 용량처럼 여러 속성이 함께 있으면 대표 속성 하나만 출력된다는 점을 확인해야 합니다.